# 03 — Synthetic Supplementary Analysis

Full learner grid, oracle-selected estimator results, per-estimator CDV advantage,
CDV support statistics, and tidy result DataFrames for further analysis.

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'synthetic'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    build_full_learner_table, ranking_metrics_within_groups, ranking_comparison_table,
    resolve_alternative, describe_paired_test,
    METHODS_ORDER, LEARNERS_ORDER
)
from helpers.plotting import METHOD_LABELS, plot_scissors_chart

results_by_alpha = {}
for alpha in CFG.ALPHA_VALUES:
    full_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    debug_path = CFG.CHECKPOINT_PATH_TEMPLATE.format(alpha=alpha)
    if os.path.exists(full_path):
        path = full_path
    else:
        path = debug_path
        print(f'WARNING: full-scale checkpoint not found for alpha={alpha:.2f}; '
              f'falling back to DEBUG checkpoint (small N_TRAIN/N_TEST, few seeds). '
              f'Run the full loop in 01_experiment.ipynb for real results.')
    results_by_alpha[alpha] = load_checkpoint(path)
    print(f'alpha={alpha:.2f}: {len(results_by_alpha[alpha])} seeds  [{os.path.basename(path)}]')


alpha=0.00: 20 seeds  [results_alpha_0.00.pkl]
alpha=0.25: 20 seeds  [results_alpha_0.25.pkl]
alpha=0.50: 20 seeds  [results_alpha_0.50.pkl]
alpha=0.75: 20 seeds  [results_alpha_0.75.pkl]
alpha=1.00: 20 seeds  [results_alpha_1.00.pkl]


## 1. Full Learner Grid per Alpha — ATE MSE

In [2]:
for alpha in CFG.ALPHA_VALUES:
    print(f'\nATE MSE — alpha={alpha:.2f}')
    table = build_full_learner_table(results_by_alpha[alpha], metric='ate_mse')
    display(table.round(5))


ATE MSE — alpha=0.00


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.00492,0.02421,1.32121,0.02530,0.00399,0.01197
GLOBAL_MISSINGNESS,0.00511,0.02388,0.00280,0.02545,0.00399,0.02235
GLOBAL_MISSINGNESS_CDV,0.00525,0.02170,0.00265,0.02339,0.00393,0.02371
MATCHED_RANDOM_PARTITIONS,0.00396,0.03252,1.32241,0.04962,0.00467,0.00659
CDV_SEPARATE,0.00739,0.00541,0.02255,0.01689,0.00409,0.01155



ATE MSE — alpha=0.25


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.00382,0.02317,1.39438,0.02577,0.00455,0.02460
GLOBAL_MISSINGNESS,0.00407,0.02285,0.00810,0.02610,0.00435,0.02955
GLOBAL_MISSINGNESS_CDV,0.00436,0.02079,0.00829,0.02372,0.00442,0.03421
MATCHED_RANDOM_PARTITIONS,0.00426,0.03121,1.39665,0.05291,0.00485,0.01363
CDV_SEPARATE,0.00614,0.00479,0.01095,0.01751,0.00520,0.01800



ATE MSE — alpha=0.50


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.00433,0.02316,1.47140,0.02740,0.00434,0.04674
GLOBAL_MISSINGNESS,0.00404,0.02303,0.02661,0.02829,0.00430,0.04093
GLOBAL_MISSINGNESS_CDV,0.00420,0.02044,0.02717,0.02485,0.00412,0.04851
MATCHED_RANDOM_PARTITIONS,0.00596,0.03044,1.47483,0.05314,0.00557,0.02748
CDV_SEPARATE,0.00640,0.00410,0.00675,0.01945,0.00492,0.03065



ATE MSE — alpha=0.75


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.00525,0.02302,1.55228,0.02766,0.00387,0.07464
GLOBAL_MISSINGNESS,0.00456,0.02353,0.05831,0.02878,0.00408,0.05506
GLOBAL_MISSINGNESS_CDV,0.00439,0.02070,0.05930,0.02473,0.00388,0.06564
MATCHED_RANDOM_PARTITIONS,0.00842,0.03114,1.55696,0.05312,0.00775,0.04536
CDV_SEPARATE,0.00790,0.00414,0.00993,0.02109,0.00457,0.04219



ATE MSE — alpha=1.00


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.00687,0.02228,1.63701,0.02640,0.00401,0.10615
GLOBAL_MISSINGNESS,0.00491,0.02261,0.10322,0.02714,0.00403,0.06450
GLOBAL_MISSINGNESS_CDV,0.00510,0.02012,0.10466,0.02304,0.00385,0.07774
MATCHED_RANDOM_PARTITIONS,0.01210,0.03060,1.64303,0.05126,0.00980,0.06858
CDV_SEPARATE,0.01172,0.00496,0.02052,0.02254,0.00473,0.05980


## 2. Full Learner Grid per Alpha — CATE MSE

In [3]:
for alpha in CFG.ALPHA_VALUES:
    print(f'\nCATE MSE — alpha={alpha:.2f}')
    table = build_full_learner_table(results_by_alpha[alpha], metric='cate_mse')
    display(table.round(5))


CATE MSE — alpha=0.00


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,0.04196,1.69566,1.32121,1.71857,0.55559,0.03548
GLOBAL_MISSINGNESS,0.07192,1.69667,0.00280,1.71660,0.55742,0.06453
GLOBAL_MISSINGNESS_CDV,0.07389,1.68456,0.05116,1.70544,0.55501,0.06954
MATCHED_RANDOM_PARTITIONS,0.04583,0.90078,1.32346,0.95436,0.27843,0.03082
CDV_SEPARATE,0.10013,1.64647,0.17955,1.78874,0.65918,0.08172



CATE MSE — alpha=0.25


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,1.14720,1.79436,2.91561,1.84622,0.71675,1.17912
GLOBAL_MISSINGNESS,0.78717,1.79256,1.52933,1.84243,0.71973,0.80238
GLOBAL_MISSINGNESS_CDV,0.67887,1.77135,1.37962,1.82284,0.71420,0.69849
MATCHED_RANDOM_PARTITIONS,1.17764,1.05593,2.91918,1.16275,0.44627,1.17540
CDV_SEPARATE,0.20828,1.73290,1.56192,1.85873,0.75966,0.18912



CATE MSE — alpha=0.50


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,4.46646,2.03547,7.55631,2.09177,0.93876,4.58766
GLOBAL_MISSINGNESS,2.93660,2.03074,6.11152,2.08330,0.93865,3.01117
GLOBAL_MISSINGNESS_CDV,2.49344,1.97616,5.32542,2.03420,0.92914,2.57049
MATCHED_RANDOM_PARTITIONS,4.53025,1.48391,7.56302,1.62886,0.88455,4.59410
CDV_SEPARATE,0.51690,1.91576,5.60228,2.01880,0.90588,0.50298



CATE MSE — alpha=0.75


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,10.00346,2.29335,15.24334,2.34489,1.16438,10.26586
GLOBAL_MISSINGNESS,6.52250,2.28801,13.74937,2.33926,1.16657,6.69243
GLOBAL_MISSINGNESS_CDV,5.51925,2.19365,11.88856,2.24492,1.14476,5.67860
MATCHED_RANDOM_PARTITIONS,10.10563,2.00454,15.25499,2.15684,1.52194,10.27208
CDV_SEPARATE,1.03427,2.16506,12.30062,2.23904,1.09960,1.01324



CATE MSE — alpha=1.00


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,17.75689,2.60841,25.97667,2.64059,1.47035,18.21722
GLOBAL_MISSINGNESS,11.54337,2.59825,24.44288,2.63093,1.47083,11.85167
GLOBAL_MISSINGNESS_CDV,9.75402,2.48610,21.06906,2.51731,1.43496,10.03444
MATCHED_RANDOM_PARTITIONS,17.90125,2.58623,25.99509,2.71865,2.31600,18.21321
CDV_SEPARATE,1.75912,2.51206,21.65695,2.49647,1.38956,1.72469


## 3. Oracle-Selected Estimator Results

In [4]:
oracle_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for seed, sr in res.items():
        for method in METHODS_ORDER:
            oracle = sr.get('oracle', {}).get(method, {})
            m = oracle.get('metrics', {})
            oracle_rows.append({
                'alpha': alpha, 'outer_seed': seed, 'method': method,
                'selected_learner': oracle.get('selected_learner', 'N/A'),
                'ate_mse': m.get('ate_mse', np.nan),
                'cate_mse': m.get('cate_mse', np.nan),
            })

oracle_df = pd.DataFrame(oracle_rows)
print('Oracle ATE MSE by alpha and method:')
display(oracle_df.groupby(['alpha', 'method'])['ate_mse'].mean().unstack().round(5))

print('\nOracle learner selection frequency per alpha and method:')
display(oracle_df.groupby(['alpha', 'method', 'selected_learner']).size().unstack(fill_value=0))

oracle_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'oracle_results.parquet'), index=False)

Oracle ATE MSE by alpha and method:


method,CDV_SEPARATE,GLOBAL_MISSINGNESS,GLOBAL_MISSINGNESS_CDV,GLOBAL_SENTINEL,MATCHED_RANDOM_PARTITIONS
alpha,,,,,
0.00,0.01904,0.01904,0.02224,0.01056,0.01056
0.25,0.00723,0.00719,0.01917,0.00448,0.00453
0.50,0.00428,0.00430,0.00408,0.00435,0.00431
0.75,0.00398,0.00407,0.00381,0.00388,0.00386
1.00,0.00406,0.00398,0.00367,0.00405,0.00403



Oracle learner selection frequency per alpha and method:


selected_learner                 DR_RF  Double_ML  X_RF
alpha method                                           
0.00  CDV_SEPARATE                   5         15     0
      GLOBAL_MISSINGNESS             5         15     0
      GLOBAL_MISSINGNESS_CDV         2         18     0
      GLOBAL_SENTINEL                3         17     0
      MATCHED_RANDOM_PARTITIONS      3         17     0
0.25  CDV_SEPARATE                   7          6     7
      GLOBAL_MISSINGNESS             7          6     7
      GLOBAL_MISSINGNESS_CDV         7         12     1
      GLOBAL_SENTINEL                0          0    20
      MATCHED_RANDOM_PARTITIONS      0          0    20
0.50  CDV_SEPARATE                   0          0    20
      GLOBAL_MISSINGNESS             0          0    20
      GLOBAL_MISSINGNESS_CDV         0          0    20
      GLOBAL_SENTINEL                0          0    20
      MATCHED_RANDOM_PARTITIONS      0          0    20
0.75  CDV_SEPARATE                   0          0    20
      GLOBAL_MISSINGNESS             0          0    20
      GLOBAL_MISSINGNESS_CDV         0          0    20
      GLOBAL_SENTINEL                0          0    20
      MATCHED_RANDOM_PARTITIONS      0          0    20
1.00  CDV_SEPARATE                   0          0    20
      GLOBAL_MISSINGNESS             0          0    20
      GLOBAL_MISSINGNESS_CDV         0          0    20
      GLOBAL_SENTINEL                0          0    20
      MATCHED_RANDOM_PARTITIONS      0          0    20

## 4. Ranking Metrics — Kendall τ and Spearman ρ (primary learner) per Alpha

Undefined (NaN) at alpha=0, where true ITE is constant and rank correlation is not identifiable.


In [5]:
print(f'Ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds, per alpha')
ranking_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        tau_vals, rho_vals = [], []
        for sr in res.values():
            m = sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {})
            tau_vals.append(m.get('kendall_tau', np.nan))
            rho_vals.append(m.get('spearman_rho', np.nan))
        ranking_rows.append({
            'alpha': alpha,
            'Method': METHOD_LABELS.get(method, method),
            'Kendall τ mean': np.nanmean(tau_vals),
            'Kendall τ std':  np.nanstd(tau_vals),
            'Spearman ρ mean': np.nanmean(rho_vals),
            'Spearman ρ std':  np.nanstd(rho_vals),
            'N seeds': int(np.sum(np.isfinite(tau_vals))),
        })

ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df.round(4))
ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics.csv'), index=False)


Ranking metrics (DR_RF) — mean ± std across seeds, per alpha


,alpha,Method,Kendall τ mean,Kendall τ std,Spearman ρ mean,Spearman ρ std,N seeds
0,0.00,Global (Sentinel),NaN,NaN,NaN,NaN,0
1,0.00,Global + Missingness,NaN,NaN,NaN,NaN,0
2,0.00,Global + Miss. + CDV ID,NaN,NaN,NaN,NaN,0
3,0.00,Matched Random Partitions,NaN,NaN,NaN,NaN,0
4,0.00,CDV Separate (proposed),NaN,NaN,NaN,NaN,0
5,0.25,Global (Sentinel),0.3056,0.0182,0.4500,0.0234,20
6,0.25,Global + Missingness,0.4098,0.0343,0.5631,0.0425,20
7,0.25,Global + Miss. + CDV ID,0.5032,0.0252,0.6818,0.0307,20
8,0.25,Matched Random Partitions,0.3025,0.0226,0.4463,0.0288,20
9,0.25,CDV Separate (proposed),0.7741,0.0204,0.9223,0.0138,20


In [6]:
selected_learner_rank = CFG.PRIMARY_LEARNER  # change to compare a different learner

print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
print(describe_paired_test('CDV_SEPARATE', 'method', 'Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    print(f'\nCDV_SEPARATE vs other methods — Kendall τ ({selected_learner_rank}), alpha={alpha:.2f}')
    tau_cmp = ranking_comparison_table(res, selected_learner_rank, metric='kendall_tau',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    tau_cmp.index = [METHOD_LABELS.get(m, m) for m in tau_cmp.index]
    display(tau_cmp.round(4))

    print(f'CDV_SEPARATE vs other methods — Spearman ρ ({selected_learner_rank}), alpha={alpha:.2f}')
    rho_cmp = ranking_comparison_table(res, selected_learner_rank, metric='spearman_rho',
                                        ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
    rho_cmp.index = [METHOD_LABELS.get(m, m) for m in rho_cmp.index]
    display(rho_cmp.round(4))


paired bootstrap CI (10,000 resamples), two-sided (alpha=0.05). Δ = CDV_SEPARATE − method  (Kendall τ), paired per outer seed. H1: Δ ≠ 0.
paired bootstrap CI (10,000 resamples), two-sided (alpha=0.05). Δ = CDV_SEPARATE − method  (Spearman ρ), paired per outer seed. H1: Δ ≠ 0.

CDV_SEPARATE vs other methods — Kendall τ (DR_RF), alpha=0.00


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
Global (Sentinel),NaN,NaN,"[nan, nan]",NaN,NaN
Global + Missingness,NaN,NaN,"[nan, nan]",NaN,NaN
Global + Miss. + CDV ID,NaN,NaN,"[nan, nan]",NaN,NaN
Matched Random Partitions,NaN,NaN,"[nan, nan]",NaN,NaN


CDV_SEPARATE vs other methods — Spearman ρ (DR_RF), alpha=0.00


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
Global (Sentinel),NaN,NaN,"[nan, nan]",NaN,NaN
Global + Missingness,NaN,NaN,"[nan, nan]",NaN,NaN
Global + Miss. + CDV ID,NaN,NaN,"[nan, nan]",NaN,NaN
Matched Random Partitions,NaN,NaN,"[nan, nan]",NaN,NaN



CDV_SEPARATE vs other methods — Kendall τ (DR_RF), alpha=0.25


...

,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
Global (Sentinel),0.4528,0.0202,"[0.4943, 0.5117]",0.0,111.0307
Global + Missingness,0.5728,0.0332,"[0.3686, 0.3978]",0.0,66.8366
Global + Miss. + CDV ID,0.7102,0.0167,"[0.2380, 0.2531]",0.0,34.5585
Matched Random Partitions,0.4526,0.0213,"[0.4942, 0.5124]",0.0,111.1674



CDV_SEPARATE vs other methods — Kendall τ (DR_RF), alpha=1.00


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
Global (Sentinel),0.3078,0.0149,"[0.5288, 0.5420]",0.0,173.9187
Global + Missingness,0.4167,0.0230,"[0.4165, 0.4366]",0.0,102.3261
Global + Miss. + CDV ID,0.5287,0.0129,"[0.3084, 0.3206]",0.0,59.4843
Matched Random Partitions,0.3074,0.0154,"[0.5290, 0.5426]",0.0,174.2724


CDV_SEPARATE vs other methods — Spearman ρ (DR_RF), alpha=1.00


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
Global (Sentinel),0.4526,0.0199,"[0.4967, 0.5134]",0.0,111.5564
Global + Missingness,0.5731,0.0300,"[0.3716, 0.3979]",0.0,67.0819
Global + Miss. + CDV ID,0.7131,0.0147,"[0.2378, 0.2510]",0.0,34.2630
Matched Random Partitions,0.4523,0.0204,"[0.4967, 0.5140]",0.0,111.7069


## 5. Within-CDV Ranking Metrics — Kendall τ and Spearman ρ (primary learner) per Alpha

The Section 4 metrics pool predictions across all CDV groups (and OTHER) before ranking.
Here, Kendall τ / Spearman ρ are computed separately WITHIN each CDV group, then
size-weighted averaged across groups — isolating within-subgroup ranking quality from
across-group ordering effects.


In [7]:
print(f'Within-CDV ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds, per alpha')
within_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for method in METHODS_ORDER:
        tau_vals, rho_vals = [], []
        for sr in res.values():
            pred = sr.get('predictions', {}).get(method, {}).get(CFG.PRIMARY_LEARNER, {})
            ite_pred = np.asarray(pred.get('ite_pred', []))
            ite_true = np.asarray(pred.get('ite_true', []))
            variant = np.asarray(pred.get('variant', []))
            if len(ite_pred) == 0:
                continue
            m = ranking_metrics_within_groups(ite_pred, ite_true, variant)
            tau_vals.append(m['kendall_tau_within'])
            rho_vals.append(m['spearman_rho_within'])
        within_rows.append({
            'alpha': alpha,
            'Method': METHOD_LABELS.get(method, method),
            'Kendall τ (within-CDV) mean': np.nanmean(tau_vals),
            'Kendall τ (within-CDV) std':  np.nanstd(tau_vals),
            'Spearman ρ (within-CDV) mean': np.nanmean(rho_vals),
            'Spearman ρ (within-CDV) std':  np.nanstd(rho_vals),
            'N seeds': int(np.sum(np.isfinite(tau_vals))),
        })

within_ranking_df = pd.DataFrame(within_rows)
display(within_ranking_df.round(4))
within_ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics_within_cdv.csv'), index=False)


Within-CDV ranking metrics (DR_RF) — mean ± std across seeds, per alpha


,alpha,Method,Kendall τ (within-CDV) mean,Kendall τ (within-CDV) std,Spearman ρ (within-CDV) mean,Spearman ρ (within-CDV) std,N seeds
0,0.00,Global (Sentinel),NaN,NaN,NaN,NaN,0
1,0.00,Global + Missingness,NaN,NaN,NaN,NaN,0
2,0.00,Global + Miss. + CDV ID,NaN,NaN,NaN,NaN,0
3,0.00,Matched Random Partitions,NaN,NaN,NaN,NaN,0
4,0.00,CDV Separate (proposed),NaN,NaN,NaN,NaN,0
5,0.25,Global (Sentinel),0.2205,0.0157,0.2861,0.0208,20
6,0.25,Global + Missingness,0.3282,0.0695,0.3986,0.1047,20
7,0.25,Global + Miss. + CDV ID,0.4196,0.0638,0.5357,0.0970,20
8,0.25,Matched Random Partitions,0.2193,0.0171,0.2844,0.0219,20
9,0.25,CDV Separate (proposed),0.7708,0.0193,0.8983,0.0159,20


## 6. Per-Estimator CDV Advantage per Alpha

In [8]:
advantage_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for learner in LEARNERS_ORDER:
        cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        gs_vals  = [sr['metrics'].get('GLOBAL_SENTINEL', {}).get(learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        cdv_mean = np.nanmean(cdv_vals)
        gs_mean  = np.nanmean(gs_vals)
        improvement_pct = (gs_mean - cdv_mean) / gs_mean * 100 if gs_mean > 0 else np.nan
        advantage_rows.append({
            'alpha': alpha, 'learner': learner,
            'cdv_ate_mse': cdv_mean, 'global_ate_mse': gs_mean,
            'improvement_pct': improvement_pct,
        })

adv_df = pd.DataFrame(advantage_rows)
print('CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha:')
display(adv_df.pivot(index='learner', columns='alpha', values='improvement_pct').round(2))

CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha:


alpha,0.00,0.25,0.50,0.75,1.00
learner,,,,,
DR_RF,-50.32,-60.84,-47.75,-50.49,-70.53
Double_ML,3.47,26.83,34.42,43.47,43.66
S_Linear,98.29,99.21,99.54,99.36,98.75
S_RF,77.64,79.34,82.29,82.02,77.74
T_RF,33.23,32.05,29.00,23.77,14.60
X_RF,-2.68,-14.22,-13.37,-17.96,-17.89


In [9]:
selected_learner = CFG.PRIMARY_LEARNER  # change to compare a different learner
baseline_methods = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']

vs_methods_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(selected_learner, {}).get('ate_mse', np.nan) for sr in res.values()]
    cdv_mean = np.nanmean(cdv_vals)
    for method in baseline_methods:
        base_vals = [sr['metrics'].get(method, {}).get(selected_learner, {}).get('ate_mse', np.nan) for sr in res.values()]
        base_mean = np.nanmean(base_vals)
        improvement_pct = (base_mean - cdv_mean) / base_mean * 100 if base_mean > 0 else np.nan
        vs_methods_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'cdv_ate_mse': cdv_mean, 'baseline_ate_mse': base_mean,
            'improvement_pct': improvement_pct,
        })

vs_methods_df = pd.DataFrame(vs_methods_rows)
print(f'CDV_SEPARATE improvement (%) over each method (learner={selected_learner}, ATE MSE):')
display(vs_methods_df.pivot(index='method', columns='alpha', values='improvement_pct').round(2))

CDV_SEPARATE improvement (%) over each method (learner=DR_RF, ATE MSE):


alpha,0.00,0.25,0.50,0.75,1.00
method,,,,,
Global (Sentinel),-50.32,-60.84,-47.75,-50.49,-70.53
Global + Miss. + CDV ID,-40.71,-40.82,-52.40,-80.21,-129.60
Global + Missingness,-44.80,-50.92,-58.44,-73.27,-138.84
Matched Random Partitions,-86.93,-44.13,-7.41,6.15,3.15


## 7. Per-Estimator CDV Advantage per Alpha — CATE

In [10]:
advantage_cate_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for learner in LEARNERS_ORDER:
        cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(learner, {}).get('cate_mse', np.nan) for sr in res.values()]
        gs_vals  = [sr['metrics'].get('GLOBAL_SENTINEL', {}).get(learner, {}).get('cate_mse', np.nan) for sr in res.values()]
        cdv_mean = np.nanmean(cdv_vals)
        gs_mean  = np.nanmean(gs_vals)
        improvement_pct = (gs_mean - cdv_mean) / gs_mean * 100 if gs_mean > 0 else np.nan
        advantage_cate_rows.append({
            'alpha': alpha, 'learner': learner,
            'cdv_cate_mse': cdv_mean, 'global_cate_mse': gs_mean,
            'improvement_pct': improvement_pct,
        })

adv_cate_df = pd.DataFrame(advantage_cate_rows)
print('CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha (CATE MSE):')
display(adv_cate_df.pivot(index='learner', columns='alpha', values='improvement_pct').round(2))

CDV_SEPARATE improvement (%) over GLOBAL_SENTINEL per learner per alpha (CATE MSE):


alpha,0.00,0.25,0.50,0.75,1.00
learner,,,,,
DR_RF,-138.60,81.84,88.43,89.66,90.09
Double_ML,-130.31,83.96,89.04,90.13,90.53
S_Linear,86.41,46.43,25.86,19.30,16.63
S_RF,2.90,3.43,5.88,5.59,3.69
T_RF,-4.08,-0.68,3.49,4.51,5.46
X_RF,-18.64,-5.99,3.50,5.56,5.49


In [11]:
selected_learner_cate = CFG.PRIMARY_LEARNER  # change to compare a different learner
baseline_methods_cate = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']

vs_methods_cate_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    cdv_vals = [sr['metrics'].get('CDV_SEPARATE', {}).get(selected_learner_cate, {}).get('cate_mse', np.nan) for sr in res.values()]
    cdv_mean = np.nanmean(cdv_vals)
    for method in baseline_methods_cate:
        base_vals = [sr['metrics'].get(method, {}).get(selected_learner_cate, {}).get('cate_mse', np.nan) for sr in res.values()]
        base_mean = np.nanmean(base_vals)
        improvement_pct = (base_mean - cdv_mean) / base_mean * 100 if base_mean > 0 else np.nan
        vs_methods_cate_rows.append({
            'alpha': alpha, 'method': METHOD_LABELS.get(method, method),
            'cdv_cate_mse': cdv_mean, 'baseline_cate_mse': base_mean,
            'improvement_pct': improvement_pct,
        })

vs_methods_cate_df = pd.DataFrame(vs_methods_cate_rows)
print(f'CDV_SEPARATE improvement (%) over each method (learner={selected_learner_cate}, CATE MSE):')
display(vs_methods_cate_df.pivot(index='method', columns='alpha', values='improvement_pct').round(2))

CDV_SEPARATE improvement (%) over each method (learner=DR_RF, CATE MSE):


alpha,0.00,0.25,0.50,0.75,1.00
method,,,,,
Global (Sentinel),-138.60,81.84,88.43,89.66,90.09
Global + Miss. + CDV ID,-35.52,69.32,79.27,81.26,81.97
Global + Missingness,-39.22,73.54,82.40,84.14,84.76
Matched Random Partitions,-118.49,82.31,88.59,89.77,90.17


## 8. Build Tidy Main Results DataFrame

In [12]:
main_rows = []
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    for seed, sr in res.items():
        for method in METHODS_ORDER:
            for learner in LEARNERS_ORDER:
                m = sr.get('metrics', {}).get(method, {}).get(learner, {})
                main_rows.append({
                    'dataset': 'synthetic',
                    'outer_seed': seed, 'alpha': alpha,
                    'method': method, 'learner': learner,
                    'ate_mse': m.get('ate_mse', np.nan),
                    'cate_mse': m.get('cate_mse', np.nan),
                    'kendall_tau': m.get('kendall_tau', np.nan),
                    'spearman_rho': m.get('spearman_rho', np.nan),
                    'n_train': sr.get('n_train', np.nan),
                    'n_test': sr.get('n_test', np.nan),
                    'n_retained_cdvs': len(sr.get('retained_cdv_info', {})),
                    'pct_other_train': sr.get('pct_other_train', np.nan),
                })

main_df = pd.DataFrame(main_rows)
out_path = os.path.join(CFG.ARTIFACTS_DIR, 'main_results.parquet')
main_df.to_parquet(out_path, index=False)
print(f'Tidy results: {main_df.shape}')
print(f'Saved: {out_path}')

Tidy results: (3000, 13)
Saved: revised_experiment/synthetic/artifacts\main_results.parquet


## 9. CDV Support Statistics per Alpha

In [13]:
for alpha in CFG.ALPHA_VALUES:
    res = results_by_alpha.get(alpha, {})
    pct_others = [sr.get('pct_other_train', np.nan) for sr in res.values()]
    n_retained = [len(sr.get('retained_cdv_info', {})) for sr in res.values()]
    print(f'alpha={alpha:.2f}: retained_CDVs={np.nanmean(n_retained):.1f}±{np.nanstd(n_retained):.1f}, '
          f'OTHER%={np.nanmean(pct_others):.1f}±{np.nanstd(pct_others):.1f}')

alpha=0.00: retained_CDVs=3.0±0.0, OTHER%=13.0±0.4
alpha=0.25: retained_CDVs=3.0±0.0, OTHER%=13.0±0.4
alpha=0.50: retained_CDVs=3.0±0.0, OTHER%=13.0±0.4
alpha=0.75: retained_CDVs=3.0±0.0, OTHER%=13.0±0.4
alpha=1.00: retained_CDVs=3.0±0.0, OTHER%=13.0±0.4
